# Phase 3: Data Preparation & Train/Validation/Test Split

**Question of this phase:** *how do we prepare the data so that the models are judged fairly?*

Two jobs:
1. **Split** the policies into three sets, like exercises and exams for a pupil.
2. **Encode** the rating factors into numbers a GLM can use, with rules learned on the training set
   only.

All the code lives in `src/features/pipeline.py`; this notebook runs it and checks the result.

## 0. Setup

In [1]:
import sys  # access to Python's module search path
from pathlib import Path  # file paths

import joblib  # saves Python objects (here, the fitted encoder) to a file
import numpy as np  # maths on columns
import pandas as pd  # tables

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # project root
sys.path.insert(0, str(PROJECT_ROOT))  # make the project's code importable

from src.utils.hand_calc import show_policy_in_dataset, steps_table  # dataset extract; step-by-step table
from src.features.pipeline import (  # the project's preparation functions
    PolicyFeatureEncoder,  # turns rating factors into numbers
    compare_split_balance,  # compares the three sets
    get_exposure_offset,  # log(Exposure), the frequency model's offset
    get_frequency_target,  # number of claims per policy
    get_severity_subset,  # policies with a paid claim
    train_val_test_split_policies,  # the 60 / 20 / 20 split
)

pd.set_option("display.max_columns", 40)  # show up to 40 columns
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")  # print numbers as 1,234.5678

df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "policy_level_clean.parquet")  # clean table from Phase 1
print(f"Loaded {len(df):,} policies.")

Loaded 668,626 policies.


## 1. Split into train / validation / test

| Set | Share | Role (pupil analogy) |
|---|---|---|
| Train | 60% | the exercises: the model learns on them |
| Validation | 20% | a first exam the model has never seen |
| Test | 20% | a second exam the model has never seen |

Claims are rare (about 3.7% of policies have one). The split is **stratified** on the claim class
(0, 1, 2+ claims) so that each set gets the same share of policies with claims.

In [2]:
train_df, val_df, test_df = train_val_test_split_policies(df, val_size=0.20, test_size=0.20, random_state=42)  # the split

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    share = len(part) / len(df)  # share of all policies
    with_claim = (part["ClaimNb_freq_target"] > 0).mean()  # share of policies with at least one claim
    print(f"{name:<11}: {len(part):>7,} policies ({share:.1%}), share with a claim {with_claim:.4%}")
print(f"\nThe three sets add back to the whole table: {len(train_df) + len(val_df) + len(test_df) == len(df)}")

Train      : 401,175 policies (60.0%), share with a claim 3.6902%
Validation : 133,725 policies (20.0%), share with a claim 3.6897%
Test       : 133,726 policies (20.0%), share with a claim 3.6904%

The three sets add back to the whole table: True


**Result:** 60% / 20% / 20%, and the same share of policies with a claim in each set (to four
decimals).

## 2. Do the three sets look like the same portfolio?

In [3]:
compare_split_balance(train_df, val_df, test_df)  # size, frequency and average claim cost of each set

,split,n_policies,share_of_total,exposure_weighted_frequency,avg_severity
0,train,401175,0.6000,0.0740,"2,005.6315"
1,validation,133725,0.2000,0.0740,"3,341.8886"
2,test,133726,0.2000,0.0741,"1,975.5157"


**Reading:** the frequency (claims / years insured) is the same in the three sets (7.40-7.41%). The
average claim cost differs: about EUR 2,006 (train), 3,342 (validation), 1,976 (test). Is the split
biased? The next cell compares the **median** claim and counts the very large claims.

In [4]:
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sev = get_severity_subset(part)  # policies with a paid claim
    n_above_100k = (sev["AvgClaimAmount"] > 100_000).sum()  # policies whose average claim exceeds EUR 100,000
    print(
        f"{name:<11} paid-claim policies={len(sev):>6,}  median=EUR {sev['AvgClaimAmount'].median():>6,.0f}  "
        f"mean=EUR {sev['AvgClaimAmount'].mean():>6,.0f}  above EUR 100k={n_above_100k:>2}  "
        f"largest=EUR {sev['AvgClaimAmount'].max():>10,.0f}"
    )

train       paid-claim policies=14,792  median=EUR  1,172  mean=EUR  1,962  above EUR 100k=22  largest=EUR    774,412
validation  paid-claim policies= 4,928  median=EUR  1,172  mean=EUR  3,293  above EUR 100k=12  largest=EUR  4,075,401
test        paid-claim policies= 4,931  median=EUR  1,172  mean=EUR  1,936  above EUR 100k= 5  largest=EUR    276,643


**Finding: the split is not biased.** The median claim is EUR 1,172 in every set (one of the standard
settlement amounts that recur thousands of times in the file). The averages differ because the
validation set happened to receive the largest claim of the dataset (EUR 4.08M) and 12 policies with
claims above EUR 100,000 on average. An average moves with a few extreme values; a median does not.
This will matter again in Phase 8.

Same check on the categories:

In [5]:
for col in ["VehGas", "VehBrand"]:  # two category columns as examples
    mix = pd.concat(  # share of each level in each set, side by side
        [
            train_df[col].value_counts(normalize=True).rename("train"),
            val_df[col].value_counts(normalize=True).rename("validation"),
            test_df[col].value_counts(normalize=True).rename("test"),
        ],
        axis=1,
    )
    print(f"\n--- {col} share by set ---")
    print(mix)


--- VehGas share by set ---
         train  validation   test
VehGas                           
Regular 0.5089      0.5068 0.5061
Diesel  0.4911      0.4932 0.4939

--- VehBrand share by set ---
          train  validation   test
VehBrand                          
B12      0.2412      0.2421 0.2422
B1       0.2411      0.2397 0.2410
B2       0.2361      0.2380 0.2363
B3       0.0794      0.0794 0.0788
B5       0.0513      0.0518 0.0515
B6       0.0424      0.0424 0.0423
B4       0.0376      0.0363 0.0376
B10      0.0263      0.0263 0.0266
B11      0.0202      0.0205 0.0200
B13      0.0183      0.0174 0.0178
B14      0.0060      0.0062 0.0058


**Result:** the same mix in the three sets, within a fraction of a percentage point.

## 3. Encode the rating factors (rules learned on train only)

| Factor | Treatment | Why |
|---|---|---|
| `DrivAge`, `BonusMalus`, `VehPower` | kept as numbers | no data problem; their shape is a modeling question (Phases 4-5) |
| `Density` | `log(Density)` | from 1 to 27,000; the log brings it to 0 to 10 |
| `VehAge` | capped at the 99.5th percentile **of train** | some ages up to 100 are not plausible |
| `VehBrand`, `VehGas`, `Region` | one 0/1 column per level, except a reference level | how a GLM uses categories |
| `Area` | dropped | copy of `Density` (Phase 2) |
| `Exposure` | **not a factor**: the offset `log(Exposure)` | time insured is a definition, not a risk factor |

**Example of a category column.** `VehGas` has two levels, Diesel and Regular. We create one column,
`VehGas_Regular`, equal to 1 for Regular and 0 for Diesel. Diesel is the reference: the GLM's
intercept describes it, and the coefficient of `VehGas_Regular` says how Regular differs.

In [6]:
encoder = PolicyFeatureEncoder()  # an empty encoder
encoder.fit(train_df)  # learns the VehAge cap and the category levels, from TRAIN ONLY

print(f"VehAge cap learned on train (99.5th percentile): {encoder.vehage_cap_:.1f} years")
print(f"Largest raw VehAge in validation              : {val_df['VehAge'].max()} (the encoder never saw it)")
print(f"Reference levels: " + ", ".join(f"{col} = {levels[0]}" for col, levels in encoder.categories_.items()))

X_train = encoder.transform(train_df)  # numbers for the GLM, train set
X_val = encoder.transform(val_df)  # same rules, validation set
X_test = encoder.transform(test_df)  # same rules, test set

print(f"\nX_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"Columns: {encoder.feature_names_}")

VehAge cap learned on train (99.5th percentile): 24.0 years
Largest raw VehAge in validation              : 100 (the encoder never saw it)
Reference levels: VehBrand = B1, VehGas = Diesel, Region = R11



X_train: 401,175 rows x 37 columns
Columns: ['DrivAge', 'BonusMalus', 'VehPower', 'log_Density', 'VehAge_capped', 'VehBrand_B10', 'VehBrand_B11', 'VehBrand_B12', 'VehBrand_B13', 'VehBrand_B14', 'VehBrand_B2', 'VehBrand_B3', 'VehBrand_B4', 'VehBrand_B5', 'VehBrand_B6', 'VehGas_Regular', 'Region_R21', 'Region_R22', 'Region_R23', 'Region_R24', 'Region_R25', 'Region_R26', 'Region_R31', 'Region_R41', 'Region_R42', 'Region_R43', 'Region_R52', 'Region_R53', 'Region_R54', 'Region_R72', 'Region_R73', 'Region_R74', 'Region_R82', 'Region_R83', 'Region_R91', 'Region_R93', 'Region_R94']


**The policies.** The first two rows of the train set (`train.parquet`), highlighted below. We follow
each of their raw values into the columns the GLM receives.

In [7]:
example = train_df.head(2)  # two policies from the train set
show_policy_in_dataset(train_df, example["IDpol"], ["DrivAge", "BonusMalus", "VehPower", "Density", "VehAge", "VehBrand", "VehGas", "Region"])

,IDpol,DrivAge,BonusMalus,VehPower,Density,VehAge,VehBrand,VehGas,Region
row,,,,,,,,,
0,4179513,40,67,7,149.0000,12,B2,Diesel,R24
1,1175230,54,50,11,48.0000,1,B14,Diesel,R24
2,80440,44,50,4,53.0000,4,B4,Diesel,R24
3,33179,34,80,6,9307.0000,2,B6,Diesel,R82


In [8]:
encoded = encoder.transform(example)  # the same two policies, encoded
non_zero = encoded.loc[:, (encoded != 0).any()].copy()  # hide the 0/1 columns that are 0 for both policies
non_zero.insert(0, "IDpol", example["IDpol"].to_numpy())  # identify each row by its policy number
print("Encoded values, as the GLM receives them:")
print(non_zero.to_string(index=False))

Encoded values, as the GLM receives them:
  IDpol  DrivAge  BonusMalus  VehPower  log_Density  VehAge_capped  VehBrand_B14  VehBrand_B2  Region_R24
4179513       40          67         7       5.0039             12        0.0000       1.0000      1.0000
1175230       54          50        11       3.8712              1        1.0000       0.0000      1.0000


**The calculation** for the first policy, column by column:

In [9]:
first = example.iloc[0]  # the first policy, raw values
first_encoded = encoded.iloc[0]  # the same policy, encoded
rows = []
for col in ["DrivAge", "BonusMalus", "VehPower"]:  # numeric columns kept as they are
    rows.append((col, f"{first[col]} kept as it is", f"{first_encoded[col]:g}", f"column {col} of the dataset"))
rows.append(("log_Density", f"ln({first['Density']:,.0f})", f"{first_encoded['log_Density']:.4f}", "column Density, logarithm"))
rows.append(("VehAge_capped", f"min({first['VehAge']}, {encoder.vehage_cap_:g})", f"{first_encoded['VehAge_capped']:g}",
             "column VehAge, capped at the train cap"))
for col in ["VehBrand", "VehGas", "Region"]:  # categories: one 0/1 column per level
    name = f"{col}_{first[col]}"
    if name in first_encoded.index:
        rows.append((name, f"{col} = {first[col]} -> 1, other {col} columns -> 0", "1", f"column {col} of the dataset"))
    else:
        rows.append((col, f"{col} = {first[col]} is the reference level -> all {col} columns 0", "0", f"column {col} of the dataset"))
print(f"IDpol {first['IDpol']}")
steps_table(rows)

IDpol 4179513


step,calculation,result,source
DrivAge,40 kept as it is,40,column DrivAge of the dataset
BonusMalus,67 kept as it is,67,column BonusMalus of the dataset
VehPower,7 kept as it is,7,column VehPower of the dataset
log_Density,ln(149),5.0039,"column Density, logarithm"
VehAge_capped,"min(12, 24)",12,"column VehAge, capped at the train cap"
VehBrand_B2,"VehBrand = B2 -> 1, other VehBrand columns -> 0",1,column VehBrand of the dataset
VehGas,VehGas = Diesel is the reference level -> all VehGas columns 0,0,column VehGas of the dataset
Region_R24,"Region = R24 -> 1, other Region columns -> 0",1,column Region of the dataset


**Reading.** Numbers go to the model as they are (or as their logarithm for the density, capped for the
vehicle age); a category becomes a 1 in the column of its level and 0 elsewhere. The reference level
has no column of its own: its effect is in the intercept, and every other level is measured against it.

**No leakage check:** the vehicle-age cap learned on train is applied unchanged to validation and test.

In [10]:
print(f"Largest VehAge_capped: train {X_train['VehAge_capped'].max()}, validation {X_val['VehAge_capped'].max()}, test {X_test['VehAge_capped'].max()}")
assert X_val["VehAge_capped"].max() <= encoder.vehage_cap_  # validation capped at the train threshold
assert X_test["VehAge_capped"].max() <= encoder.vehage_cap_  # test capped at the train threshold
assert X_train.isnull().sum().sum() == 0 and X_val.isnull().sum().sum() == 0 and X_test.isnull().sum().sum() == 0  # no empty cell
print("All three sets use the train cap; no missing value after encoding.")

Largest VehAge_capped: train 24, validation 24, test 24


All three sets use the train cap; no missing value after encoding.


## 4. Target, offset and severity subset

In [11]:
y_freq_train = get_frequency_target(train_df)  # number of claims per policy
offset_train = get_exposure_offset(train_df)  # log(Exposure)
print("Frequency target (first 5 policies):", y_freq_train.head().tolist())
print(f"Offset log(Exposure): from {offset_train.min():.3f} to {offset_train.max():.3f} (0.0 = a full year)")

sev_train = get_severity_subset(train_df)  # policies with a paid claim
print(f"\nSeverity training set: {len(sev_train):,} policies with a paid claim (out of {len(train_df):,})")

Frequency target (first 5 policies): [1, 0, 0, 0, 0]
Offset log(Exposure): from -5.903 to 0.000 (0.0 = a full year)

Severity training set: 14,792 policies with a paid claim (out of 401,175)


## 5. Save the three sets and the encoder

In [12]:
processed_dir = PROJECT_ROOT / "data" / "processed"  # folder for the processed tables
models_dir = PROJECT_ROOT / "models"  # folder for fitted objects
models_dir.mkdir(parents=True, exist_ok=True)  # create it if needed

train_df.to_parquet(processed_dir / "train.parquet", index=False)  # save the train set
val_df.to_parquet(processed_dir / "val.parquet", index=False)  # save the validation set
test_df.to_parquet(processed_dir / "test.parquet", index=False)  # save the test set
joblib.dump(encoder, models_dir / "feature_encoder.joblib")  # save the fitted encoder

print("Saved: data/processed/train.parquet, val.parquet, test.parquet and models/feature_encoder.joblib")

Saved: data/processed/train.parquet, val.parquet, test.parquet and models/feature_encoder.joblib


## Phase 3 summary

| Check | Result |
|---|---|
| Split | 60% / 20% / 20% (401,175 / 133,725 / 133,726 policies) |
| Share of policies with a claim | identical to four decimals in the three sets |
| Frequency per set | 7.40% / 7.40% / 7.41% |
| Median claim per set | EUR 1,172 in all three (averages differ because of a few very large claims) |
| Category mix | identical within a fraction of a point |
| Leakage | VehAge cap learned on train only, applied unchanged elsewhere |

**Next:** Phase 4, the frequency model.